In [ ]:
import cygnet as cy
import numpy as np
import uproot, gc, os
import imageio.v3 as iio
import matplotlib.pyplot as plt

# folders:
# input
# mask
# mask_exp2
# mask_exp4

metric_test_path = "/home/frx/dataset/image_dataset/metric_test"

data_path = "/home/frx/dataset/root_dataset"

folders = os.listdir(data_path)
files_dict = {}

for fol in folders:
    sub_path = os.path.join(data_path, fol)
    files_dict[fol] = sorted(os.listdir(sub_path))

display(files_dict)

ufile_sample = uproot.open(data_path + "/" + fol + "/" + files_dict[fol][0])
tree = ufile_sample["event_info"]

n_classi = len(folders)
n_file = len(files_dict[fol])
n_eventi = len(tree["eventnumber"].arrays(library="np")["eventnumber"])

print(f"i see {n_classi} class folders each containing {n_file} files, each containing {n_eventi} events.")

i = 0
j = 0
k = 0

train = 0
vali = 0
test = 0

tot = n_classi * n_file * n_eventi
print(f"i see a total of {tot} events")
printable = []

ffol = folders[0]
fffile = files_dict[ffol][0]

factor = 4
exp_radius = 4

in_t = cy.input_transform_builder(factor)
out_t = cy.target_transform_builder(factor)
exp2_t = cy.inflate_transform_builder(2)
exp4_t = cy.inflate_transform_builder(4)

dry_run = True
if not dry_run:
	for fol, files in files_dict.items():

		for j, ffile in enumerate(files):
			if j < 3:
				train_set = True
			else:
				train_set = False

			fpath = os.path.join(data_path, fol, ffile)
			with uproot.open(fpath) as ufile:
				tree = ufile["event_info"]
				redxs = tree["redpix_ix"].arrays(library="np")["redpix_ix"]
				redys = tree["redpix_iy"].arrays(library="np")["redpix_iy"]
				redzs = tree["redpix_iz"].arrays(library="np")["redpix_iz"]
				p_shape = ufile[f"pic_run{j+1}_ev{0}"].to_numpy()[0].shape

				for k in range(n_eventi):
					print(fol, ffile, train, vali, test)
					noisy = ufile[f"pic_run{j+1}_ev{k}"].to_numpy()[0].T
					xe = redxs[k]
					ye = redys[k]
					mask = np.zeros(p_shape, dtype=np.uint8)
					mask[xe, ye] = redzs
					mask = mask.T

					noisy = in_t(noisy).squeeze()
					mask = out_t(mask).squeeze()
					exp2_mask = exp2_t(mask).squeeze()
					exp4_mask = exp4_t(mask).squeeze()

					if train_set:
						train += 1
						continue

					else:
						if k < 100:
							vali += 1
							continue
						else:
							# iio.imwrite(os.path.join(test_path, "input", f"i_{test}.png"), noisy, extension=".png")
							# iio.imwrite(os.path.join(test_path, "mask", f"m_{test}.png"), mask, extension=".png")
							# iio.imwrite(os.path.join(test_path, "mask_exp2", f"m2_{test}.png"), exp2_mask, extension=".png")
							# iio.imwrite(os.path.join(test_path, "mask_exp4", f"m4_{test}.png"), exp4_mask, extension=".png")
							test += 1

					noisy = None
					mask = None
					exp2_mask = None
					exp4_mask = None
					gc.collect()

				tree = None
				redxs = None
				redys = None
				redzs = None
				gc.collect()
else:
	print("dry!")

print()
print(f"N eventi: {tot}")
print(f"dataset split: {train} / {vali} / {test}")
print(f"o, in proporzione: {train/tot} / {vali/tot} / {test/tot}")